# Rigol / FPGA TDC V16: 5 MHz finite bursts and edge-drift diagnostics

Requires **VERSION16 / ID0x54444310** and the updated API/SCPI server.
Build externally, enter the actual routed WNS and bitstream label below, then
restart the kernel and run from the top with fresh RISE/FALL calibration.
V16 has no routed timing or hardware qualification yet. V15 WNS (-0.023 ns)
must not be attributed to this build. Results remain provisional until qualified.

V16 processes 32 taps per encoder clock, separates the failed transition-span
subtraction/comparison, and admits a 200 ns common period (5 MHz) at the existing
250 MHz sampling clock. This changes burst throughput; it does not promise finer
physical resolution. The 4 ns coarse tick and six-field width records remain.

AWG period = FPGA launch period = acquisition frame duration. Frame offset shifts
the full frame in 4 ns steps; it does not phase-lock the source. Retries screen
only for complete frames. All attempts and samples are retained; jitter and bias
never select a preferred capture.

V16 saves both internal edge ticks for records 1,65,129,... across each burst.
Together with raw codes and fresh LUTs, these show relative RISE/FALL drift and
check coarse/fine association. They are not an independent pin-width reference.
No smoothing, data trimming, historical LUT reuse or automatic width correction
is applied. Broad distributions and narrow-pulse bias remain open until measured.


## 1. Configuration

Use the `envscpi` kernel and fill the actual bitstream label and routed WNS.
VERSION identifies the interface, not a unique placed image. A new implementation
needs new fine LUTs; unchanged hardware does not need rebuilding for this notebook.

All frequency values below are **requests for a common rate**. For example,
997003 Hz requests 1003.006 ns, which becomes 1004 ns / 996015.936 Hz on both
instruments. Requested/applied rates and AWG/FPGA readbacks are saved explicitly.
Period readback tolerance describes instrument resolution, not physical clock lock.
No duty-dependent shortening or deliberate AWG/frame detuning is permitted.

`MEASURED_RETURN_DELAY_NS` is an optional scope observation from the FPGA launch
edge to input RISE. It is logged only; V14+ does not auto-convert that observation into FRAME_OFFSET_NS or provide a calibrated pin-delay reference. V16 adds sparse internal edge ticks. Configure the frame offset explicitly. Relative phase and
jitter/drift must also be characterized. The target is 100% complete frames after
alignment; low yield before alignment is a diagnostic result, not a predicted loss.

AWG width requests can quantize by 0.1 ns or five significant digits. Width
analysis uses readback, or `REFERENCE_OVERRIDES_NS` for independent scope widths.
Setting resolution is not physical accuracy. Reference overrides remain keyed by
the requested HIGH duration. No scope instrument is automated here.

The current global routed timing still needs acceptance; use the actual report
for this image. This notebook never changes saved historical capture metadata.

**Upgrading an open V14 notebook:** restart the kernel and run from the
configuration cell onward. The setup cell reloads the helper, uses the configured expected version
and creates a new results directory; an existing `bench` retains its old
configuration. V15 preflight also requires the updated API/SCPI `OFFSET?`
command. Do not combine earlier-build captures or LUTs with this run.


In [1]:
# If needed, uncomment and run in this notebook's kernel, then restart it:
# %pip install numpy matplotlib pyvisa
# PyVISA also needs the vendor VISA runtime with USB support (already used by discovery).

RP_IP = "169.254.77.151"
SCPI_PORT = 5000
VISA_RESOURCE = "USB0::0x1AB1::0x0647::DG5R270700006::INSTR"
VISA_BACKEND = ""           # Default installed vendor VISA runtime.
AWG_COMMAND_DELAY_S = 0.02  # Space USB commands, including error-queue queries.
AWG_CHANNEL = 1
AWG_SERIAL = "DG5R270700006"
LOW_V, HIGH_V = 0.0, 3.3
EXPECTED_WIDTH_VERSION = 16
BITSTREAM_LABEL = "v16-5MHz: enter actual build identity"
IMPLEMENTATION_WNS_NS = None  # Enter the NEW V16 routed WNS; do not reuse V15.

FRAME_COUNT = 8192           # One nominal source cycle per acquisition frame.
SWEEP_FRAMES = 4096
HOST_TIMEOUT_S = 30.0
BASELINE_FREQUENCY_HZ = 997_003.0
BASELINE_WIDTH_NS = 500.25   # Fractional 4 ns tick: exposes interpolation improvement.
CAL_FREQUENCIES_HZ = [997_003.0, 991_027.0, 983_039.0]  # Both instruments follow each common rate.
CAL_ROUNDS = 8               # 24 bursts; usable count depends on alignment.
MIN_EDGE_SAMPLES = 50_000
MIN_POPULATED_BINS = 32
KNOWN_DEAD_ZONES = []        # Physically characterized dead bins; keep unresolved gaps separate.
# Recharacterize the NEW build first. Old v9 exceptions were FALL 91-93 / 99-101.
PROVISIONAL_MISSING_CODE_RANGES = {}
REFERENCE_OVERRIDES_NS = {}  # Example: {500.25: 500.18}; use measured widths, in ns.
RESULTS_PARENT = "tdc_rigol_results"
ALIGNMENT_SURVEY_REPEATS = 4
ALIGNMENT_SURVEY_FRAMES = 1024
MEASURED_RETURN_DELAY_NS = None  # Independent scope observation; does not auto-set offset.
FRAME_OFFSET_NS = 0              # V14+: shift frame boundary; choose from scope phase measurement.
FRAME_OFFSETS_BY_PERIOD_NS = {}   # Optional {common_period_ns: offset_ns}, overrides global setting.
# Offset is a multiple of 4 ns, 0..262140 ns, strictly less than each common period.
# Example ONLY: for T=1004, input rise=800/fall=1500 ns, offset=600 ns encloses the pulse.
# This helper restarts sources each capture. Unlocked AWG phase is not made repeatable by OFFSET.
TARGET_COMPLETE_FRACTION = 1.0   # Require every requested pulse in the selected burst.
PHASE_RETRIES = True            # Restart incomplete pulse bursts, including 0 complete.
PHASE_RETRY_MAX_ATTEMPTS = 50    # Per test; prevents an endless loop with absent/bad input.
PHASE_RETRY_DELAY_S = 0.2        # Delay after failed attempt; settings stay unchanged.
DRIFT_REPEATS = 3  # Frozen image/LUT; keep every repeat regardless of width SD/bias.


In [2]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Works when Jupyter starts either in the repository root or in scpi-tests.
candidates = [Path.cwd(), Path.cwd()/"scpi-tests"]
helper_dir = next((p for p in candidates if (p/"tdc_rigol_tools.py").is_file()), None)
if helper_dir is None:
    raise RuntimeError("Start Jupyter in RedPitaya_app or scpi-tests; keep tdc_rigol_tools.py beside this notebook")
sys.path.insert(0, str(helper_dir.resolve()))
# Reload edited helper code when rerunning this cell in an existing kernel.
import importlib
import tdc_rigol_tools
importlib.reload(tdc_rigol_tools)
from tdc_rigol_tools import (Bench, save_json, summarize, rows, quality_guard,
                            build_candidate, plot_capture, plot_candidate, plot_sweep,
                            precision_comparison, fit_width_reference,
                            validate_width_reference, phase_residuals, signal_identity_check,
                            plan_matched_period, alignment_survey)

# Use the configuration cell as the single source of the expected version.
if EXPECTED_WIDTH_VERSION not in (14, 15, 16):
    raise ValueError("Configure expected VERSION14, VERSION15 or VERSION16, then rerun setup")
FRAME_OFFSET_NS = globals().get("FRAME_OFFSET_NS", 0)
FRAME_OFFSETS_BY_PERIOD_NS = globals().get("FRAME_OFFSETS_BY_PERIOD_NS", {})
PHASE_RETRIES = globals().get("PHASE_RETRIES", True)
PHASE_RETRY_MAX_ATTEMPTS = globals().get("PHASE_RETRY_MAX_ATTEMPTS", 50)
PHASE_RETRY_DELAY_S = globals().get("PHASE_RETRY_DELAY_S", 0.2)
config = dict(expected_width_version=EXPECTED_WIDTH_VERSION, rp_ip=RP_IP, scpi_port=SCPI_PORT, visa_resource=VISA_RESOURCE,
              visa_backend=VISA_BACKEND, awg_channel=AWG_CHANNEL, awg_serial=AWG_SERIAL,
              low_v=LOW_V, high_v=HIGH_V, bitstream_label=BITSTREAM_LABEL,
              implementation_wns_ns=IMPLEMENTATION_WNS_NS, frame_count=FRAME_COUNT,
              host_timeout_s=HOST_TIMEOUT_S, socket_timeout_s=10.0,
              visa_timeout_ms=10000, awg_command_delay_s=AWG_COMMAND_DELAY_S, settle_s=0.05, timing_policy="matched_period_v1",
              measured_return_delay_ns=MEASURED_RETURN_DELAY_NS,
              frame_offset_ns=FRAME_OFFSET_NS, frame_offsets_by_period_ns=FRAME_OFFSETS_BY_PERIOD_NS,
              phase_retries=PHASE_RETRIES, phase_retry_max_attempts=PHASE_RETRY_MAX_ATTEMPTS,
              phase_retry_delay_s=PHASE_RETRY_DELAY_S)
from local_data import results_directory
RUN_DIR = results_directory(RESULTS_PARENT)/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
bench = Bench(config, RUN_DIR)
save_json(RUN_DIR/"test_plan.json", dict(cal_frequencies_hz=CAL_FREQUENCIES_HZ,
          cal_rounds=CAL_ROUNDS, min_edge_samples=MIN_EDGE_SAMPLES,
          min_populated_bins=MIN_POPULATED_BINS, known_dead_zones=KNOWN_DEAD_ZONES,
          provisional_missing_code_ranges=PROVISIONAL_MISSING_CODE_RANGES,
          reference_overrides_ns=REFERENCE_OVERRIDES_NS, timing_policy="matched_period_v1",
          target_complete_fraction=TARGET_COMPLETE_FRACTION,
          measured_return_delay_ns=MEASURED_RETURN_DELAY_NS))

def reference_for(width):
    return REFERENCE_OVERRIDES_NS.get(float(width))

def take(label, frequency, width, *, fine=False, frames=SWEEP_FRAMES):
    capture = bench.acquire(label, frequency, float(width), frames=frames, fine=fine,
                            reference_ns=reference_for(width))
    settings, timing, summary = capture["awg"], capture["timing"], capture["summary"]
    print(f"Requested common rate {frequency:g} Hz; applied AWG/frame "
          f"{timing['common_frequency_hz']:.9g} Hz / {timing['common_period_ns']} ns")
    print(f"Readbacks: AWG period {settings['period_ns']:.9g} ns, "
          f"FPGA period {timing['fpga_period_readback_ns']} ns; "
          f"AWG HIGH {settings['width_ns']:.9g} ns (requested {width:g})")
    print(f"{label}: {summary['complete_pulses']}/{summary['total_frames']} complete "
          f"({summary['complete_fraction']:.1%}); operating target 100% after alignment; "
          f"phase lock unverified; RISE/FALL {summary['fine_edge_quality']}; "
          f"FPGA-fine calibrated {summary['calibrated_pulses']}")
    retry = capture.get("phase_retry")
    if retry:
        print(f"Selected source start {retry['attempt']}/{retry['max_attempts']}; "
              f"complete fraction across all starts: {retry['complete_fraction_all_attempts']:.1%}; "
              f"ledger: {retry['ledger_path']}")
    if "edge_diagnostics" in capture:
        d = capture["edge_diagnostics"]
        print("Sparse edge pairs:", d["valid_pairs"], "/", d["sampled"],
              "; RISE/FALL drift span (ns):",
              d["rise_drift"]["range_ns"], d["fall_drift"]["range_ns"])
    return capture


def show_and_save(fig, name):
    fig.savefig(RUN_DIR/name, dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)

print("Notebook requires TDC VERSION:", EXPECTED_WIDTH_VERSION)
print("Helper loaded from:", Path(tdc_rigol_tools.__file__).resolve())
print("Python:", sys.executable)
print("Results:", RUN_DIR.resolve())
if IMPLEMENTATION_WNS_NS is None or IMPLEMENTATION_WNS_NS < 0:
    print("PROVISIONAL: implementation timing is unknown or failed; calibration cannot repair this.")
BASELINE_PLAN = plan_matched_period(BASELINE_FREQUENCY_HZ)
print("Common baseline timing:", BASELINE_PLAN)
print("Equal programmed periods; phase lock and return-delay calibration remain unverified.")
print("Incomplete pulse-burst restarts:", PHASE_RETRIES, "; maximum starts per test:", PHASE_RETRY_MAX_ATTEMPTS)


Notebook requires TDC VERSION: 16
Helper loaded from: C:\Users\kairan\vscodeprj\RedPitaya_app\scpi-tests\tdc_rigol_tools.py
Python: c:\Users\kairan\miniconda3\envs\envscpi\python.exe
Results: C:\Users\kairan\RedPitaya_local_data\RedPitaya_app\scpi-tests\tdc_rigol_results\20261023T142035_329621Z
PROVISIONAL: implementation timing is unknown or failed; calibration cannot repair this.
Common baseline timing: {'policy': 'matched_period_v1', 'requested_frequency_hz': 997003.0, 'requested_period_ns': 1003.006009009, 'common_period_ns': 1004, 'common_frequency_hz': 996015.93625498, 'grid_ns': 4, 'period_rounding_ns': 0.9939909910000324, 'phase_lock_verified': False, 'note': 'Both instruments receive the same nominal period. Relative phase, clock drift and launch-to-return delay are not established by equal settings.'}
Equal programmed periods; phase lock and return-delay calibration remain unverified.
Incomplete pulse-burst restarts: True ; maximum starts per test: 50


In [3]:
# An old Bench keeps its original version/config even after editing a cell.
if bench.config.get("expected_width_version") != EXPECTED_WIDTH_VERSION:
    raise RuntimeError("Bench configuration is stale. Restart the kernel and run configuration/setup from the top.")
# Outputs are stopped; verifies the configured version (V16 by default), 512 taps, 4000 ps coarse tick and capacity.
# No FPGA reset: existing tables are preserved and backed up before any upload.
PREFLIGHT = bench.preflight()
print(json.dumps(PREFLIGHT, indent=2))
print("Preflight passed. Each acquisition will configure/enable CH1, then stop both outputs.")
MINIMUM_PERIOD_NS = PREFLIGHT["image"]["minimum_period_ns"]
print("Minimum common frame period (ns):", MINIMUM_PERIOD_NS)


RuntimeError: Load the matching bitstream: expected VERSION=16, TICK=4000, TAPS=512; {'width_version': 17, 'tick_ps': 4000, 'taps': 512, 'capacity': 16384, 'bitstream_label': 'v16-5MHz: enter actual build identity', 'implementation_wns_ns': None}

## 1b. Restart-to-complete screening

Each point restarts the same nominal common-period settings until a complete
burst is received (maximum PHASE_RETRY_MAX_ATTEMPTS). All failed attempts are
retained. Inspect phase_retry.attempt and complete_fraction_all_attempts;
100% in the selected burst is the selection criterion, not measured phase lock.
This does not calibrate return delay or the frame offset.


In [ ]:
ALIGNMENT_SURVEY, alignment_captures = alignment_survey(
    bench, BASELINE_FREQUENCY_HZ, BASELINE_WIDTH_NS,
    repeats=ALIGNMENT_SURVEY_REPEATS, frames=ALIGNMENT_SURVEY_FRAMES)
print(json.dumps(ALIGNMENT_SURVEY, indent=2))
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, len(ALIGNMENT_SURVEY["points"])+1),
        [p["complete_fraction"] for p in ALIGNMENT_SURVEY["points"]], "o-")
ax.axhline(TARGET_COMPLETE_FRACTION, color="C3", linestyle="--", label="Operating target after alignment")
ax.set(xlabel="Test point (each may require multiple starts)", ylabel="Selected-burst complete fraction", ylim=(-0.02, 1.02))
ax.grid(alpha=0.3); ax.legend()
show_and_save(fig, "matched_alignment_survey.png")
print("V14+ applies the configured frame offset saved in each capture. Verify launch-to-input phase/drift on the scope; OFFSET does not lock the AWG.")


## 2. Baseline and frame fit, fine correction OFF

The FPGA frame period equals its launch period. The AWG is programmed to that
same nominal period, regardless of pulse width. A valid operating frame contains
one return HIGH with the correct launch association and no extra edge.

After alignment, aim for one valid measurement per frame. With the current
free-running AWG, phase/drift can make yield high, low or variable across restarts.
Do not label 50% or another geometric fraction as the expected operating yield.

Interpret separately: complete-frame yield; RISE/FALL code validity **among complete
frames**; mean width error; width standard deviation; and individual status flags.
A complete or fine-valid frame does not prove width accuracy. Inspect boundary,
HIGH_START, NO_RISE/NO_FALL and EXTRA_EDGES counts during alignment.
The existing ABI reports width, not the input RISE time relative to launch, so it
cannot itself measure the cable/VTC delay. MEASURED_RETURN_DELAY_NS is logged only; FRAME_OFFSET_NS is programmed and read back for each acquisition.


In [ ]:
baseline = take("baseline_coarse", BASELINE_FREQUENCY_HZ, BASELINE_WIDTH_NS,
                frames=FRAME_COUNT)
print(json.dumps(baseline["summary"], indent=2))
print("Record integrity:", baseline["record_integrity"])
show_and_save(plot_capture(baseline), "baseline.png")
quality_guard(baseline)  # Blocks calibration of no-signal/glitch-only captures.

## 2b. Verify which signal the FPGA is measuring

The original baseline is near 500 ns, which is also the FPGA window output's HIGH duration.
One matching width cannot prove the source. This check keeps the common AWG/FPGA period fixed for every width and the FPGA
launch output at 50% duty. Read the actual period and launch HIGH from
`SIGNAL_IDENTITY`; the default 997003 Hz request becomes a **1004 ns** common
period and **502 ns** launch HIGH. The acquisition frame is the whole period,
not just the launch output HIGH. Only the AWG HIGH duration changes:

1. **Driven DC 0 V:** AWG output ON at a defined LOW, with the internal FPGA windows still
   running. Expect zero complete pulses. This avoids leaving the input floating with AWG OFF.
2. **100.2, 300.2 and 700.2 ns HIGH:** the measured widths should follow these changes,
   with coarse quantization and a possible fixed timing offset. A result fixed near the FPGA launch HIGH
   indicates that the intended stimulus has not been established.

All raw records and source readbacks are saved. Calibration is blocked if DC LOW
produces complete pulses, measured widths do not track the source, or a width
point has too few complete frames. Inspect `SIGNAL_IDENTITY["errors"]` to
distinguish these conditions. Zero complete frames have no measured mean;
they do not establish that the FPGA measured a different signal. In v12 and v14,
`HIGH_START | EXTRA_EDGES | NO_FALL | SATURATED` is consistent with a pulse
cut by the frame boundary. With a 1004 ns frame, a 700.2 ns pulse has only
about 303.8 ns of available RISE placement before allowing for detection
offsets and margins. The helper restarts both sources for every point, so
passing the 100.2/300.2 ns points does not preserve alignment at 700.2 ns.
Keep equal periods. With phase retries enabled, each width point uses the first
fully complete burst after source restarts; every failed start is saved in its
retry ledger. This demonstrates stimulus identity conditional on frame fit,
not phase lock. Instrument/integrity failures and DC false positives remain faults.

For actual DC/width mismatch failures, Check CH1 vs CH2, E1 pin 7,
common return, unintended connections and coupling; then inspect capture/timing logic if the
scope confirms the correct input. This check does not uniquely distinguish wiring from HDL faults.
The driven-DC command follows the Rigol programming guide linked above.

**How to read the baseline plot:** its two main horizontal levels are 4 ns coarse width codes,
not a reconstruction of the input waveform. Missing markers are rejected acquisition frames.
Clusters of accepted/rejected frames arise from the beat between independent clocks. The
`SATURATED` flag here means the frame ended HIGH without a captured FALL; it does not mean
analogue clipping. Fine correction is OFF during baseline and identity tests.


**DC readback compatibility:** live queries on this generator confirmed `FUNCtion? = ARB`, `APPLy? = DC,...`, `FUNCtion:ARBitrary? = DC` and zero offset, with CH1 OFF. The helper uses `APPLy:DC` and verifies the specific waveform. The redundant `FUNCtion DC` command was removed after a queue-overflow error was reported at that step. An arbitrary waveform at zero offset alone is still rejected.

USB commands are spaced by `AWG_COMMAND_DELAY_S` (default 20 ms). Every error-queue reply is logged; pre-existing errors are recorded and drained at connection, and newly reported errors still stop the acquisition after a bounded queue drain. The helper does not ignore overflow or clear new errors with `*CLS`. Reload the notebook, rerun setup/import and preflight to recreate `bench`, then rerun section 2b.


In [ ]:
SIGNAL_IDENTITY, identity_captures = signal_identity_check(
    bench, BASELINE_FREQUENCY_HZ, frames=SWEEP_FRAMES)
print(json.dumps(SIGNAL_IDENTITY, indent=2))
points = SIGNAL_IDENTITY["points"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
x = [p["readback_ns"] for p in points]
y = [np.nan if p["measured_ns"] is None else p["measured_ns"] for p in points]
axes[0].plot(x, y, "o-", label="Measured coarse mean")
axes[0].plot(x, x, "--", color="0.5", label="Ideal width tracking")
axes[0].axhline(SIGNAL_IDENTITY["internal_window_high_ns"], color="C3", linestyle=":", label="FPGA launch HIGH")
axes[0].set(xlabel="AWG HIGH readback (ns)", ylabel="Measured HIGH (ns)")
axes[0].legend()
axes[1].plot(x, [a-b for a,b in zip(y,x)], "o-")
axes[1].set(xlabel="AWG HIGH readback (ns)", ylabel="Mean error (ns)")
for ax in axes: ax.grid(alpha=0.3)
show_and_save(fig, "signal_identity.png")
if not SIGNAL_IDENTITY["passed"]:
    details = "\n".join(SIGNAL_IDENTITY.get("errors", []))
    raise RuntimeError("Signal identity qualification failed:\n" + details
                       + "\nInspect signal_identity.json and raw captures before calibrating.")


## 3. Fine calibration with equal AWG/frame periods

Each frequency request selects one common representable period for both instruments.
The three settings below change **both** rates together; they never detune source
against acquisition. A 50% AWG HIGH is computed from the actual common period.

Code density requires an effectively uniform distribution of edge phase across
4 ns. Matching repetition periods does not provide that distribution. A future
controlled AWG phase/input-delay sweep, or measured independent-clock phase drift,
must establish coverage. Shifting only a frame boundary does not sweep the input
edge relative to the sampling clock. Software output restarts are not a known phase sweep.

The current capture can proceed with both internal clocks free-running, but its
phase uniformity remains an assumption assessed by the unchanged sample, occupied-bin,
validity, concentration and cross-setting/time CDF gates. Fixed/coherent phases may
fail those gates; do not repair that by making frame and AWG rates different.

For accepted histogram h, the LUT uses bin centres over the 4000 ps tick. This
calibrates within-clock interpolation, not launch-to-return delay or absolute width.
Rise/fall tables remain independent; long pulses are used to avoid narrow-pulse bias.


In [ ]:
if not globals().get("SIGNAL_IDENTITY", {}).get("passed", False):
    raise RuntimeError("Run section 2b and resolve its signal identity failures before calibration")
if CAL_ROUNDS < 2 or len({plan_matched_period(f)["common_period_ns"] for f in CAL_FREQUENCIES_HZ}) < 2:
    raise ValueError("Use >=2 rounds at >=2 distinct common AWG/FPGA periods")
calibration_captures = []
for round_index in range(CAL_ROUNDS):
    for frequency in CAL_FREQUENCIES_HZ:
        width = 0.5*plan_matched_period(frequency)["common_period_ns"]
        capture = take(f"density_r{round_index:02d}_f{frequency:g}", frequency, width,
                       frames=FRAME_COUNT)
        calibration_captures.append(capture)
        print(f"Calibration progress: {len(calibration_captures)}/{CAL_ROUNDS*len(CAL_FREQUENCIES_HZ)} bursts")

candidate = build_candidate(calibration_captures, MIN_EDGE_SAMPLES,
                            MIN_POPULATED_BINS, KNOWN_DEAD_ZONES,
                            provisional_missing_code_ranges=PROVISIONAL_MISSING_CODE_RANGES)
save_json(RUN_DIR/"candidate_luts.json", candidate)
print(json.dumps(candidate["quality"], indent=2))
print("Qualified for upload:", candidate["ready"])
show_and_save(plot_candidate(candidate), "code_density_and_luts.png")

## 4. Upload and verify both fine LUTs

This cell checks the gates, backs up the previous RISE/FALL tables, writes both
candidates in chunks, reads every entry back, and activates correction. A failed
upload attempts to restore the previous tables. No HDL/API/server rebuild is needed.

The tables are tied to the placed bitstream and input path. They reside in volatile
FPGA state; loading a bitstream loses them. The JSON export preserves the numbers
and measurement context, but a different build requires new calibration.

Use fresh density captures for each newly implemented bitstream. The old v9 FALL gaps
91-93 and 99-101 were acknowledged only for that image. V11 starts with no exceptions.
If a new run has repeatable gaps, log and explicitly configure its edge-specific
`PROVISIONAL_MISSING_CODE_RANGES` before requalification. Other gates and invalid-edge
rejection remain active. A provisional accepted-edge LUT may have phase-dependent bias.
The cell below reuses data in this kernel without contacting instruments; the next cell
backs up, uploads and reads back both LUTs. Then run section 5 for fresh validation.


In [ ]:
# Requalify captures from this matched-period run; no historical period-policy mixing.
# For a new image, rerun configuration/preflight and collect fresh density data first.
import importlib
import tdc_rigol_tools
build_candidate = importlib.reload(tdc_rigol_tools).build_candidate
# Uses the configured exceptions for this image, empty by default.
if not globals().get("SIGNAL_IDENTITY", {}).get("passed", False):
    raise RuntimeError("Source identity must pass before requalification")
if not globals().get("calibration_captures"):
    raise RuntimeError("Calibration captures are not in this kernel; rerun section 3")
if any(c.get("timing", {}).get("policy") != "matched_period_v1" for c in calibration_captures):
    raise RuntimeError("Restart the kernel and collect this matched-period workflow; historical captures remain saved")
candidate = build_candidate(calibration_captures, MIN_EDGE_SAMPLES,
                            MIN_POPULATED_BINS, KNOWN_DEAD_ZONES,
                            provisional_missing_code_ranges=PROVISIONAL_MISSING_CODE_RANGES)
if IMPLEMENTATION_WNS_NS is None or IMPLEMENTATION_WNS_NS < 0:
    candidate["limitations"].append("Implementation timing is unknown or failed; calibration cannot repair setup failures.")
save_json(RUN_DIR/"candidate_luts.json", candidate)
save_json(RUN_DIR/"provisional_calibration_issue.json", dict(
    image=candidate["image"], bitstream_label=BITSTREAM_LABEL,
    implementation_wns_ns=IMPLEMENTATION_WNS_NS,
    acknowledged_missing_code_ranges=PROVISIONAL_MISSING_CODE_RANGES,
    quality=candidate["quality"], limitations=candidate["limitations"],
    decision="Matched-period candidate; retain all quality gates and phase-coverage limitations",
    next_action="Upload with backup/readback, then fresh coarse/fine validation; inspect excluded-event fractions and phase residuals"))
print("PROVISIONAL:", candidate["limitations"])
print(json.dumps(candidate["quality"], indent=2))
print("Eligible for provisional upload:", candidate["ready"])
show_and_save(plot_candidate(candidate), "code_density_and_luts.png")


In [ ]:
bench.upload(candidate)  # Raises before modifying tables if any quality gate failed.
print("Both tables read back exactly; fine correction activated.")
print("Calibration file:", RUN_DIR/"uploaded_luts.json")

## 5. Fresh coarse/fine validation and phase dependence

None of these captures were used to construct the LUTs. Alternate OFF/ON captures
under identical source conditions. Compare records with **both fine codes valid**
in each mode; report excluded records and coarse fallback separately.

A useful calibration reduces RMS width variation across several conditions, preserves
the expected width transfer and high edge-valid fraction, and does not introduce
code-dependent bias. The mean can remain offset because LUT bin calibration does
not calibrate external input buffers or differential rise/fall path delays.

Each fine capture also checks `width_ps = 4000*coarse_ticks + RISE_LUT - FALL_LUT`
against every hardware-calibrated record. Previously unseen raw codes are counted.

The phase plot uses calibrated RISE propagation time as an **inferred phase coordinate**.
It is not an independent phase measurement. Large periodic residuals can expose clock
boundary/coarse alignment errors. Sparse phase bins or frequency-dependent shifts
make a low overall standard deviation less convincing.

In [ ]:
validation_pairs, precision_results, phase_results = [], [], []
validation_periods = [BASELINE_PLAN["common_period_ns"] + d for d in (-4, 0, 4)
                      if BASELINE_PLAN["common_period_ns"] + d >= MINIMUM_PERIOD_NS]
for frequency in [1e9/period for period in validation_periods]:
    coarse = take(f"fresh_coarse_f{frequency:g}", frequency, BASELINE_WIDTH_NS, frames=FRAME_COUNT)
    fine = take(f"fresh_fine_f{frequency:g}", frequency, BASELINE_WIDTH_NS,
                fine=True, frames=FRAME_COUNT)
    pair = precision_comparison(coarse, fine)
    pair["frequency_hz"] = fine["timing"]["common_frequency_hz"]
    pair["common_period_ns"] = fine["timing"]["common_period_ns"]
    pair["unseen_codes"] = fine["previously_unseen_codes"]
    precision_results.append(pair)
    validation_pairs.append((coarse, fine))
    print(json.dumps(pair, indent=2))
    for c in (coarse, fine):
        phase_results.append(dict(label=c["label"], fine=c["fine_requested"],
                                  frequency_hz=frequency, bins=phase_residuals(c, candidate)))
save_json(RUN_DIR/"fresh_precision_comparison.json", precision_results)
save_json(RUN_DIR/"phase_residuals.json", phase_results)
show_and_save(plot_capture(validation_pairs[1][1]), "fresh_fine.png")

fig, ax = plt.subplots(figsize=(10, 4))
for result in phase_results:
    ax.plot([b["phase_bin"] for b in result["bins"]],
            [np.nan if b["mean_error_ns"] is None else b["mean_error_ns"] for b in result["bins"]],
            ".-", label=result["label"])
ax.set(xlabel="Inferred RISE propagation-time bin across 4 ns", ylabel="Mean width minus reference (ns)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
show_and_save(fig, "phase_dependence.png")

## 6. Width range and small steps at one common repetition period

HIGH varies while the AWG and FPGA periods remain equal and fixed at the chosen
baseline period. Inspect alignment/yield across widths before interpreting the
mean or jitter: a return that crosses a frame boundary needs delay/phase analysis.
Do not shorten the frame to make the yield resemble a geometric prediction.

The 4.2 ns AWG minimum does not certify accurate 4.2 ns TDC measurements; the earlier
v12 sweep showed substantial short-pulse bias. Track width error and fine-validity
alongside yield. The 0.25 ns requested steps become quantized AWG readbacks; averaged
tracking is different from single-pulse resolution. Independent pin-width references
are required to separate input-path/stimulus effects from FPGA errors.


In [ ]:
narrow_captures = []
for width in [4.2, 5.0, 6.0, 8.0, 12.0, 20.0, 50.0, 100.0, 500.0, 900.0]:
    for fine_mode in (False, True):
        narrow_captures.append(take(f"narrow_w{width:g}_{'fine' if fine_mode else 'coarse'}",
                                    BASELINE_FREQUENCY_HZ, width, fine=fine_mode))
save_json(RUN_DIR/"narrow_sweep_summary.json", [c["summary"] for c in narrow_captures])
show_and_save(plot_sweep(narrow_captures), "minimum_width_sweep.png")
# No zero-pulse exception: failure to detect a narrow pulse is itself a result.

In [ ]:
step_captures = []
for width in np.arange(100.0, 104.01, 0.25):
    for fine_mode in (False, True):
        step_captures.append(take(f"step_w{width:g}_{'fine' if fine_mode else 'coarse'}",
                                  BASELINE_FREQUENCY_HZ, float(width), fine=fine_mode))
save_json(RUN_DIR/"small_step_summary.json", [c["summary"] for c in step_captures])
show_and_save(plot_sweep(step_captures), "small_width_steps.png")
# Error staying near a constant while requested width changes indicates tracking;
# a staircase or periodic error suggests residual quantisation/nonlinearity.

## 7. Common frequency / duty coverage

For each rate, snap to one FPGA-representable period and apply it to both instruments.
Duty changes AWG HIGH only; it never changes frame period at that rate. The measured
FPGA launch remains 50% duty as required by the current acquisition API.

Report complete frames per launch (100% operating target after alignment), rejected
frame flags, fine validity and width accuracy/repeatability. Do not claim phase lock
or a per-cycle loss rate before the source/launch association is established.
This is a finite-burst test, not continuous streaming/readout throughput.


In [ ]:
rate_captures = []
for frequency in [100_003.0, 499_979.0, 997_003.0, 1_499_977.0, 1_997_003.0,
                  3_000_000.0, 4_000_000.0, 5_000_000.0]:
    if 1e9/frequency < MINIMUM_PERIOD_NS:
        print("Unsupported by this image:", frequency)
        continue
    for duty in [10.0, 50.0, 90.0]:
        width = plan_matched_period(frequency, MINIMUM_PERIOD_NS)["common_period_ns"]*duty/100
        rate_captures.append(take(f"rate_f{frequency:g}_d{duty:g}", frequency, width, fine=True))
save_json(RUN_DIR/"rate_duty_summary.json", [dict(c["summary"], awg=c["awg"], window_ns=c["window_ns"], timing=c["timing"])
                                            for c in rate_captures])
for c in rate_captures:
    s = c["summary"]
    print(f"{c['awg']['frequency_hz']/1e3:.3f} kHz, duty {c['awg']['duty_percent']:.1f}%: "
          f"complete {s['complete_fraction']:.1%} / target 100% after alignment; phase unverified, "
          f"fine RMS {s['calibrated_only']['std_ns']} ns")

### Requests above 5 MHz

V16 admits a 200 ns common period. The 3/4 MHz requests snap to 332/252 ns (3.012/3.968 MHz);
5 MHz is exactly 200 ns. HIGH and LOW must each remain at least 8 ns for the
sampled-level detector; the 10/50/90% duty sweep satisfies this at 5 MHz.
10/50 MHz remain unsupported and are rejected here without instrument access.


In [ ]:
overspeed_captures = []  # Historical name; no mismatched-rate acquisitions are made.
for frequency in [10e6, 50e6]:
    try:
        print(plan_matched_period(frequency, MINIMUM_PERIOD_NS))
    except ValueError as exc:
        print(f"{frequency/1e6:g} MHz common-rate benchmark unavailable: {exc}")


## 7b. Repeatability and per-edge drift with one frozen calibration

Repeat previously unstable widths without selecting by SD or mean error, then
repeat 5 MHz duty points and acquire one full-capacity 5 MHz burst. All samples,
failed source starts, exclusions and sparse edge pairs remain saved. The plot
shows chronological phase changes relative to each capture's first valid pair.
Both edge traces moving together suggests common source/clock/path motion;
differential motion changes width. This alone cannot identify physical causality.
Sparse sampling can miss fast excursions, so inspect full width distributions too.


In [ ]:
drift_captures = []
for repeat in range(DRIFT_REPEATS):
    for width in [102.25, 500.25]:
        drift_captures.append(take(f"drift_r{repeat}_base_w{width:g}",
                                  BASELINE_FREQUENCY_HZ, width, fine=True))
    if MINIMUM_PERIOD_NS <= 200:
        for width in [20.25, 100.25, 180.0]:
            drift_captures.append(take(f"drift_r{repeat}_5MHz_w{width:g}",
                                      5e6, width, fine=True))
if MINIMUM_PERIOD_NS <= 200:
    drift_captures.append(take("full_capacity_5MHz", 5e6, 100.25,
                              fine=True, frames=PREFLIGHT["image"]["capacity"]))
save_json(RUN_DIR/"drift_repeat_summary.json", [dict(label=c["label"],
          summary=c["summary"], edge_diagnostics=c.get("edge_diagnostics"))
          for c in drift_captures])
for c in drift_captures:
    d = c.get("edge_diagnostics")
    print(c["label"], json.dumps(c["summary"]["calibrated_only"]))
    if d and d["valid_pairs"]:
        fig, ax = plt.subplots(figsize=(9, 3))
        ax.plot(d["frame_ids"], d["rise_phase_ns"], ".-", label="RISE")
        ax.plot(d["frame_ids"], d["fall_phase_ns"], ".-", label="FALL")
        ax.set(xlabel="Original frame ID (every 64th record)",
               ylabel="Relative edge phase (ns)", title=c["label"])
        ax.legend(); ax.grid(alpha=.3)
        show_and_save(fig, c["label"]+"_edge_drift.png")


## 8. Separate width-reference calibration: stable-capture host offset

Fine code density calibrates the within-clock interpolators. A second calibration
maps measured width onto a reference: `reference = scale * measured + offset`.
Screen five complete captures, then acquire **four different widths** for validation.
Default to scale=1 and fit offset only from widths >=100 ns, full SD <=150 ps,
and eight-block mean span <=250 ps. Report every excluded training capture;
never trim records or retry because jitter/bias is large. Affine fitting is
opt-in and requires at least three stable training widths. These gates are
diagnostic choices, not certified hardware specifications.
No validation data are fitted. This correction is saved for host use; it does not
alter the uploaded fine LUTs or silently alter raw CSV records.

With AWG readback as reference this is a nominal transfer correction, not a traceable
timebase calibration: generator timebase error, output threshold effects and TDC
scale cannot be separated. Add scope overrides if independently measured widths
are available. One width identifies offset only; multiple widths are needed for scale.

Look for a scale near 1, a repeatable offset, and small errors at the held-out widths.
Residual curvature indicates nonlinearity that an affine correction cannot remove.
A fitted scale cannot repair missing pulses or random jitter.

In [ ]:
training_captures = [take(f"reference_train_w{w:g}", BASELINE_FREQUENCY_HZ, w, fine=True)
                     for w in [20.25, 100.25, 250.25, 500.25, 800.25]]
width_model = fit_width_reference(training_captures, model="offset")
save_json(RUN_DIR/"host_width_reference_model.json", width_model)
print(json.dumps(width_model, indent=2))

holdout_captures = [take(f"reference_holdout_w{w:g}", BASELINE_FREQUENCY_HZ, w, fine=True)
                    for w in [40.25, 150.25, 350.25, 650.25]]
width_validation = validate_width_reference(width_model, holdout_captures)
save_json(RUN_DIR/"host_width_reference_validation.json", width_validation)
print(json.dumps(width_validation, indent=2))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([r["reference_ns"] for r in width_validation], [r["raw_error_ns"] for r in width_validation], ".-", label="Raw fine")
ax.plot([r["reference_ns"] for r in width_validation], [r["corrected_error_ns"] for r in width_validation], ".-", label="Host corrected")
ax.axhline(0, color="0.5"); ax.legend(); ax.grid(alpha=0.3)
ax.set(xlabel="Held-out reference HIGH (ns)", ylabel="Mean error (ns)")
show_and_save(fig, "host_reference_holdout.png")

## 9. Interpretation

| Observation | Meaning / next action |
|---|---|
| Different AWG/frame programmed periods | Configuration error; do not acquire. |
| Equal nominal periods | Frequency settings match; physical drift/phase lock remain unverified. |
| Frame yield below 100% | Inspect relative phase, launch-to-return latency, boundary crossings and missing/extra responses. Do not justify it using the old asynchronous-window prediction. |
| High yield across restarts | Observed frame fit, not proof of clock lock or measured return delay. |
| 100% selected-burst yield with retries | Required for selection; report rejected starts and aggregate complete fraction from retry ledgers. |
| Integrity/arithmetic failure | Stop and diagnose; raw records are retained. |
| Low code-density coverage | Establish sampling-clock phase coverage while preserving equal repetition periods. |
| Fine-valid fraction high | Codes passed structural checks; width accuracy can still be biased. |
| Host width correction helps | Reference transfer improved; it does not synchronize launches or reduce random jitter. |

V14 exposes a fixed frame-offset register, but does not expose launch-relative
RISE/FALL timestamps. Scope measurements and future HDL diagnostics must determine fixed
latency, drift, jitter and launch association. A constant offset in measured HIGH is
not itself the launch-to-RISE delay.

Code-density calibration, frame/launch synchronization and width-reference correction
are different quantities. Keep their data, assumptions and qualification separate.
The returned width variation includes AWG/interconnect/input/TDC effects. This test
does not establish PCB voltage conversion, ADC ENOB or 50 MS/s streaming operation.


In [ ]:
report = dict(image=PREFLIGHT["image"], timing_policy="matched_period_v1",
              alignment_survey=ALIGNMENT_SURVEY, phase_lock_verified=False,
              phase_retries=PHASE_RETRIES, phase_retry_max_attempts=PHASE_RETRY_MAX_ATTEMPTS,
              results_conditioned_on_complete_bursts=PHASE_RETRIES,
              retry_ledgers=[str(p) for p in sorted(RUN_DIR.glob("*_retries.json"))],
              measured_return_delay_ns=MEASURED_RETURN_DELAY_NS, candidate_quality=candidate["quality"],
              calibration_status=candidate["status"],
              provisional_missing_code_ranges=candidate.get("provisional_missing_code_ranges", {}),
              calibration_limitations=candidate.get("limitations", []),
              fresh_precision=precision_results, host_width_model=width_model,
              drift_repeat_summaries=[dict(label=c["label"], summary=c["summary"],
                  edge_diagnostics=c.get("edge_diagnostics")) for c in drift_captures],
              held_out_width_validation=width_validation,
              timing_closed=IMPLEMENTATION_WNS_NS is not None and IMPLEMENTATION_WNS_NS >= 0,
              note="No absolute accuracy/ADC ENOB/continuous 50 MS/s claim. Inspect all quality metrics and reference uncertainty.")
save_json(RUN_DIR/"run_report.json", report)
print("Run report:", RUN_DIR/"run_report.json")
print(bench.stop())

## STOP / recover

Each acquisition has `finally` cleanup for both outputs, including KeyboardInterrupt.
After interrupting a cell, wait for the kernel to become available and run the cell
below. It makes fresh connections and tries both instruments independently.
If the kernel or a link is unavailable, use the Rigol front-panel channel OFF button
and stop/reconnect the FPGA separately. A broken connection cannot guarantee OFF.

STOP leaves uploaded LUT contents in place but stops AWG output, FPGA window output
and acquisition. Later fine captures explicitly activate the tables again. If you
reload the FPGA, start a fresh notebook run and recalibrate.

In [ ]:
# Independent of whether baseline/calibration/sweep cells completed.
print(bench.stop())